# Primera prueba de justificación de coautoría

Esta libreta prepara un caso y pide una justificación a un LLM local en Ollama. Empieza con `C001` y Qwen2.5 7B. Ejecuta las celdas en orden desde la carpeta donde están `casos.json` e `inventario_22_llm.csv`.

## 1. Elegir el caso, el modelo y el idioma

Para repetir la prueba en inglés, se cambia `IDIOMA` a `"en"` y vuelve a ejecutar las celdas.

In [1]:
import os
import csv
import json
import time
from copy import deepcopy
from datetime import datetime, timezone

import requests

ID_CASO = "C001"
ID_MODELO = "M01"
IDIOMA = "es"

# Cargar casos
with open("casos.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

caso = None
for fila in datos["casos"]:
    if fila["id_caso"] == ID_CASO:
        caso = fila
        break

# Cargar inventario de modelos
modelo = None
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    for fila in csv.DictReader(f):
        if fila["id_modelo"] == ID_MODELO:
            modelo = fila
            break

instruccion_idioma = datos["idiomas"]["instruccion_por_idioma"][IDIOMA]

print(ID_CASO, "|", modelo["nombre_modelo"], "|", IDIOMA)
print("Motivo de selección:", caso["motivo_de_seleccion"])

C001 | Qwen2.5 7B | es
Motivo de selección: Both articles use Gaussian mixture models. The case tests whether the response identifies that specific common method and describes its role in each application.


## 2. Revisar qué recibió el par de autores

Si el caso trae probabilidades, se aplica el umbral del método o el umbral general. Si aún no hay probabilidades, solo se muestra el aviso.

In [2]:
# Se hace una copia para no modificar el caso original
entrada = deepcopy(caso["entrada_llm"])
prediccion = entrada["prediccion_coautoria"]

print("Clasificación conjunta observada:", prediccion["clasificacion_conjunta_observada"])

umbral_general = datos["regla_coautoria"]["umbral_definitivo"]

for metodo in prediccion["resultados_por_metodo"]:
    p = metodo.get("probabilidad_coautoria")
    umbral = metodo.get("umbral_aplicado")
    if umbral is None:
        umbral = umbral_general

    metodo["umbral_aplicado"] = umbral
    if p is not None and umbral is not None:
        metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
    else:
        metodo["clasificacion_por_umbral"] = None

if len(prediccion["resultados_por_metodo"]) == 0:
    print("Probabilidades por método pendientes. No se aplicó un umbral.")

similitud = entrada["comparacion_bibliografica"]["similitud_coseno"]
if similitud is None:
    print("Similitud por palabras clave: indefinida.")
else:
    print(f"Similitud por palabras clave: {similitud:.0%}")

Clasificación conjunta observada: favorable
Probabilidades por método pendientes. No se aplicó un umbral.
Similitud por palabras clave: 20%


## 3. Preparar lo que leerá el LLM

El mensaje incluye la clasificación, los artículos del caso y la comparación bibliográfica.

In [3]:
instrucciones = """Genera una justificación semántica de la predicción de coautoría recibida.
Conserva el resultado de la predicción. Si faltan probabilidades por método, indícalo.
Revisa la similitud entre los artículos y el contenido de sus títulos, resúmenes y palabras clave.
Identifica coincidencias y diferencias entre las publicaciones de ambos autores.
Cita el identificador de cada artículo que respalde tus afirmaciones.
Si la evidencia es insuficiente, indica por qué.
Distingue la similitud entre artículos de la probabilidad de coautoría.
No inventes información ni atribuyas causas al método de predicción.
Utiliza únicamente la información proporcionada."""

solicitud = {
    "model": modelo["etiqueta_ollama"],
    "messages": [
        {"role": "system", "content": instrucciones + "\n" + instruccion_idioma},
        {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
    ],
    "stream": False,
    # Parámetros de generación para la prueba piloto
    "options": {
        "temperature": 0,
        "num_ctx": 4096,
        "num_predict": 1024
    }
}

print("--- SYSTEM ---")
print(solicitud["messages"][0]["content"])
print("\n--- USER ---")
print(solicitud["messages"][1]["content"])

--- SYSTEM ---
Genera una justificación semántica de la predicción de coautoría recibida.
Conserva el resultado de la predicción. Si faltan probabilidades por método, indícalo.
Revisa la similitud entre los artículos y el contenido de sus títulos, resúmenes y palabras clave.
Identifica coincidencias y diferencias entre las publicaciones de ambos autores.
Cita el identificador de cada artículo que respalde tus afirmaciones.
Si la evidencia es insuficiente, indica por qué.
Distingue la similitud entre artículos de la probabilidad de coautoría.
No inventes información ni atribuyas causas al método de predicción.
Utiliza únicamente la información proporcionada.
Redacta la justificación en español. Cita los artículos por su identificador local.

--- USER ---
{
  "prediccion_coautoria": {
    "clasificacion_conjunta_observada": "favorable",
    "resultados_por_metodo": [],
    "mensaje_original_es": "Es posible que los autores si colaboren en un futuro"
  },
  "publicaciones": [
    {
      

## 4. Pedir una justificación a Ollama

Aquí se define la función que consulta la versión de Ollama.

In [4]:
URL_OLLAMA = "http://localhost:11434/api"

def hacer_prueba(solicitud):
    registro = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "version_ollama": "",
        "digest_modelo": "",
        "solicitud_json": json.dumps(solicitud, ensure_ascii=False),
        "estado": "error",
        "tiempo_s": "",
        "respuesta": "",
        "error": "",
        "evaluacion": "",
        "motivo_evaluacion": ""
    }

    t_inicio = None
    try:
        # Consultar versión de Ollama y digest del modelo
        r_ver = requests.get(URL_OLLAMA + "/version", timeout=10)
        r_ver.raise_for_status()
        registro["version_ollama"] = r_ver.json()["version"]

        r_tags = requests.get(URL_OLLAMA + "/tags", timeout=10)
        r_tags.raise_for_status()
        for local in r_tags.json()["models"]:
            if local["name"].lower() == solicitud["model"].lower():
                registro["digest_modelo"] = local.get("digest", "")
                break

        # Enviar solicitud al modelo
        t_inicio = time.time()
        r_chat = requests.post(URL_OLLAMA + "/chat", json=solicitud, timeout=900)
        if not r_chat.ok:
            registro["error"] = f"Ollama devolvió HTTP {r_chat.status_code}: {r_chat.text}"
        else:
            salida = r_chat.json()
            texto = salida["message"]["content"]
            registro["respuesta"] = texto

            if salida.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif salida.get("done") and texto.strip() != "":
                registro["estado"] = "completa"
            else:
                registro["estado"] = "incompleta"

    except Exception as e:
        registro["error"] = str(e)

    if t_inicio is not None:
        registro["tiempo_s"] = round(time.time() - t_inicio, 2)

    return registro

## 5. Guardar el intento y revisar la respuesta

Cada vez que se ejecuta esta celda se hace una petición a Ollama y se agrega una fila a `resultados.csv`. Se registra el idioma de la justificación. Si el archivo ya existe con los encabezados anteriores, se agrega la columna `idioma` sin eliminar los intentos guardados.

In [5]:
archivo_resultados = "resultados.csv"
columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "fecha_utc", "version_ollama", "digest_modelo",
    "solicitud_json", "estado", "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Leer los intentos anteriores, si existen
existe = os.path.exists(archivo_resultados)
intento = 1

if existe:
    with open(archivo_resultados, "r", encoding="utf-8-sig", newline="") as f:
        lector = csv.DictReader(f)
        encabezados = lector.fieldnames
        filas = list(lector)

    # Agregar el idioma al CSV anterior sin perder sus resultados
    if encabezados == [c for c in columnas if c != "idioma"]:
        for fila in filas:
            fila["idioma"] = ""
            # Recuperar el idioma de la instrucción guardada, si se puede identificar
            for idioma, instruccion in datos["idiomas"]["instruccion_por_idioma"].items():
                if instruccion in fila["solicitud_json"]:
                    fila["idioma"] = idioma
                    break

        with open(archivo_resultados, "w", encoding="utf-8-sig", newline="") as f:
            escritor = csv.DictWriter(f, fieldnames=columnas)
            escritor.writeheader()
            escritor.writerows(filas)
    elif encabezados != columnas:
        raise ValueError("Revisa los encabezados de resultados.csv antes de continuar.")

    for fila in filas:
        if fila["id_caso"] == ID_CASO and fila["id_modelo"] == ID_MODELO:
            num = int(fila["intento"])
            if num >= intento:
                intento = num + 1

# Ejecutar la prueba y guardar la fila
registro = hacer_prueba(solicitud)
registro["id_caso"] = ID_CASO
registro["id_modelo"] = ID_MODELO
registro["idioma"] = IDIOMA
registro["intento"] = intento

with open(archivo_resultados, "a", encoding="utf-8-sig", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=columnas)
    if not existe:
        escritor.writeheader()
    escritor.writerow(registro)

print("Intento:", intento, "| Idioma:", IDIOMA, "| Estado:", registro["estado"], "| Tiempo (s):", registro["tiempo_s"])
print(registro["error"] if registro["error"] != "" else registro["respuesta"])

Intento: 1 | Idioma: es | Estado: completa | Tiempo (s): 88.46
La justificación semántica de la predicción de coautoría entre los autores A01 y A02 se basa en la revisión de la similitud entre sus artículos y el contenido de sus títulos, resúmenes y palabras clave. A continuación, se presenta una justificación detallada:

1. **Similitud entre Títulos y Resúmenes**:
   - **Artículo P01**: "Ultrasound Fetal Brain Registration Using Weighted Coherent Point Drift"
   - **Artículo P02**: "Design and testing of a corpus for forensic speaker recognition using MFCC GMM and MLE"
   
   Los títulos y resúmenes de ambos artículos presentan temas muy diferentes. P01 se centra en la registración de imágenes de ultrasonido fetal, mientras que P02 se enfoca en la identificación de hablantes forense. No hay similitud temática evidente entre los títulos y resúmenes.

2. **Similitud entre Palabras Clave**:
   - Se realizó una comparación de las palabras clave de ambos artículos utilizando el método de s

### Revisión de la justificación

Revisa la respuesta junto con los artículos y se anota la evaluacion en las columnas `evaluacion` y `motivo_evaluacion` de `resultados.csv`.

In [6]:
referencia = caso["referencia_evaluacion"]
print("Artículos de referencia:", ", ".join(referencia["articulos"]))
print("Afirmación respaldada:", referencia["afirmacion_respaldada"])
print("Límite de la evidencia:", referencia["limite"])

Artículos de referencia: P01, P02
Afirmación respaldada: Both articles use Gaussian mixture models. P01 uses a Gaussian mixture model to determine the center of a point set during ultrasound image registration. P02 uses Gaussian mixture models for voice parametrization and maximum likelihood estimation for classification.
Límite de la evidencia: The texts document a shared method across different applications. They do not establish which information drove the predictor or supply a probability of coauthorship.
